# Week 1 Data Exploration

Loads the generated CSVs and displays class balance, fraud ring sizes, fraud-vs-legitimate tabular distributions, and shared-attribute frequency.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from data_generation import config
from data_generation.run_all import generate_all

raw = ROOT / 'data' / 'raw'
if not (raw / config.OUTPUT_FILES['customers']).exists():
    generate_all()

customers = pd.read_csv(raw / config.OUTPUT_FILES['customers'])
links = pd.read_csv(raw / config.OUTPUT_FILES['links'])
rings = pd.read_csv(raw / config.OUTPUT_FILES['ring_ground_truth'])
customers.head()

ModuleNotFoundError: No module named 'faker'

In [ ]:
def bars(series, title):
    counts = series.value_counts().sort_index()
    width, bar_height, gap, left = 720, 26, 8, 170
    height = 52 + len(counts) * (bar_height + gap)
    max_count = max(counts.max(), 1)
    rows = [f'<h3>{title}</h3><svg width="{width}" height="{height}" role="img">']
    for i, (label, count) in enumerate(counts.items()):
        y = 32 + i * (bar_height + gap)
        bar_width = int((width - left - 70) * count / max_count)
        rows.append(f'<text x="0" y="{y + 18}" font-size="13">{label}</text>')
        rows.append(f'<rect x="{left}" y="{y}" width="{bar_width}" height="{bar_height}" fill="#326771"></rect>')
        rows.append(f'<text x="{left + bar_width + 8}" y="{y + 18}" font-size="13">{count}</text>')
    rows.append('</svg>')
    display(HTML(''.join(rows)))

bars(customers['is_synthetic_fraud'].map({config.LEGITIMATE_LABEL: 'legitimate', config.FRAUD_LABEL: 'synthetic_fraud'}), 'Class Balance')

In [ ]:
ring_sizes = rings.groupby('ring_id').size()
bars(ring_sizes, 'Ring Size Distribution')

In [ ]:
def histogram_compare(column, bins, title):
    legit = customers.loc[customers['is_synthetic_fraud'].eq(config.LEGITIMATE_LABEL), column]
    fraud = customers.loc[customers['is_synthetic_fraud'].eq(config.FRAUD_LABEL), column]
    edges = pd.cut(pd.concat([legit, fraud]), bins=bins, retbins=True)[1]
    frame = pd.DataFrame({
        'legitimate': pd.cut(legit, bins=edges).value_counts(sort=False),
        'fraud': pd.cut(fraud, bins=edges).value_counts(sort=False),
    }).fillna(0)
    width, height, left, bottom = 760, 320, 50, 275
    plot_width = width - left - 30
    max_count = max(frame.max().max(), 1)
    bin_width = plot_width / len(frame)
    rows = [f'<h3>{title}</h3><svg width="{width}" height="{height}" role="img">']
    for i, (_, row) in enumerate(frame.iterrows()):
        x = left + i * bin_width
        legit_h = int(row['legitimate'] / max_count * 220)
        fraud_h = int(row['fraud'] / max_count * 220)
        rows.append(f'<rect x="{x:.1f}" y="{bottom - legit_h}" width="{bin_width/2 - 1:.1f}" height="{legit_h}" fill="#326771"></rect>')
        rows.append(f'<rect x="{x + bin_width/2:.1f}" y="{bottom - fraud_h}" width="{bin_width/2 - 1:.1f}" height="{fraud_h}" fill="#d95d39"></rect>')
    rows.append('<text x="50" y="305" font-size="13" fill="#326771">legitimate</text>')
    rows.append('<text x="150" y="305" font-size="13" fill="#d95d39">synthetic fraud</text>')
    rows.append('</svg>')
    display(HTML(''.join(rows)))

histogram_compare('annual_income', config.INCOME_HISTOGRAM_BINS, 'Income Distribution')
histogram_compare('credit_score', config.CREDIT_SCORE_HISTOGRAM_BINS, 'Credit Score Distribution')

In [ ]:
fraud_ids = set(customers.loc[customers['is_synthetic_fraud'].eq(config.FRAUD_LABEL), 'customer_id'])
sharing = links[links['customer_id'].isin(fraud_ids)].groupby(['attribute_type', 'attribute_id']).size()
shared_frequency = sharing[sharing.gt(1)].reset_index().groupby('attribute_type').size()
bars(shared_frequency, 'Shared Attribute Frequency Among Fraud Rings')